In [ ]:

!pip install -q transformers[torch] datasets evaluate scikit-learn pandas accelerate


In [ ]:
# Cell 2: Baseline NN Implementation
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

# 1. Load benchmark dataset & baseline tokenizer
dataset = load_dataset("dair-ai/emotion")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 2. Vectorize inputs (Convert to simple integer token IDs up to length 50)
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=50)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "label"])

train_loader = torch.utils.data.DataLoader(tokenized["train"], batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(tokenized["test"], batch_size=64, shuffle=False)

# 3. Simple Feed-Forward Network Architecture
class SimpleBaselineNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, num_classes=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(50 * embedding_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1) # Flatten out the sequence completely
        x = self.relu(self.fc1(x))
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model = SimpleBaselineNN(tokenizer.vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)

# 4. Training Loop
print("--- Training Stage 1: Simple Baseline NN ---")
baseline_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        inputs, targets = batch["input_ids"].to(device), batch["label"].to(device)
        optimizer.zero_grad()
        outputs = baseline_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 | Train Loss: {total_loss/len(train_loader):.4f}")

# 5. Evaluate Baseline
baseline_model.eval()
preds, true_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch["input_ids"].to(device)
        outputs = baseline_model(inputs)
        preds.extend(outputs.argmax(dim=-1).cpu().numpy())
        true_labels.extend(batch["label"].numpy())

print("\n--- Baseline NN Classification Report ---")
print(classification_report(true_labels, preds, target_names=dataset["train"].features["label"].names))


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

... (8 outputs left out)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

--- Training Stage 1: Simple Baseline NN ---
Epoch 1/3 | Train Loss: 1.5942
Epoch 2/3 | Train Loss: 1.4645
Epoch 3/3 | Train Loss: 1.2509

--- Baseline NN Classification Report ---
              precision    recall  f1-score   support

     sadness       0.45      0.75      0.56       581
         joy       0.57      0.66      0.61       695
        love       0.28      0.04      0.08       159
       anger       0.27      0.09      0.14       275
        fear       0.43      0.21      0.29       224
    surprise       0.00      0.00      0.00        66

    accuracy                           0.49      2000
   macro avg       0.33      0.29      0.28      2000
weighted avg       0.44      0.49      0.43      2000



In [ ]:
# Cell 3: Bidirectional GRU with Mixed Precision
import torch.cuda.amp as amp

class AdvancedBiGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.bigru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=2)
        self.fc = nn.Linear(hidden_dim * 2, num_classes) # *2 due to Bidirectional context

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.bigru(embedded)
        # Take final hidden state sequence aggregation step
        out = out[:, -1, :]
        return self.fc(out)

bigru_model = AdvancedBiGRU(tokenizer.vocab_size).to(device)
optimizer = optim.Adam(bigru_model.parameters(), lr=0.0005)
scaler = amp.GradScaler(enabled=(device.type == "cuda"))

print("--- Training Stage 2: Advanced BiGRU with AMP ---")
bigru_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in train_loader:
        inputs, targets = batch["input_ids"].to(device), batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)

        # Mixed Precision execution
        with amp.autocast(enabled=(device.type == "cuda")):
            logits = bigru_model(inputs)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 | Train Loss: {total_loss/len(train_loader):.4f}")

# Evaluate BiGRU
bigru_model.eval()
gru_preds = []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch["input_ids"].to(device)
        outputs = bigru_model(inputs)
        gru_preds.extend(outputs.argmax(dim=-1).cpu().numpy())

print("\n--- Advanced BiGRU Classification Report ---")
print(classification_report(true_labels, gru_preds, target_names=dataset["train"].features["label"].names))


/tmp/ipykernel_1952/111258472.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_1952/111258472.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=(device.type == "cuda")):


--- Training Stage 2: Advanced BiGRU with AMP ---
Epoch 1/3 | Train Loss: 1.5852
Epoch 2/3 | Train Loss: 1.5107
Epoch 3/3 | Train Loss: 1.0074

--- Advanced BiGRU Classification Report ---
              precision    recall  f1-score   support

     sadness       0.77      0.92      0.84       581
         joy       0.74      0.92      0.82       695
        love       0.00      0.00      0.00       159
       anger       0.44      0.12      0.19       275
        fear       0.45      0.73      0.55       224
    surprise       0.00      0.00      0.00        66

    accuracy                           0.68      2000
   macro avg       0.40      0.45      0.40      2000
weighted avg       0.59      0.68      0.62      2000



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Cell 4: Transformer Fine-Tuning Block
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

# 1. Load evaluation metrics mapping
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Instantiating a Pre-trained Transformer Node
transformer_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
).to(device)

# 3. Setup Native Training Pipeline Hyperparameters
training_args = TrainingArguments(
    output_dir="./transformer_results",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=(device.type == "cuda"), # Keep consistency with your project skillsets
    report_to="none"
)

# 4. Fire up Hugging Face Trainer API engine
trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("--- Training Stage 3: Fine-Tuning DistilBERT Transformer ---")
trainer.train()

# 5. Extract Final Evaluation Metrics
transformer_preds = trainer.predict(tokenized["test"])
final_transformer_labels = np.argmax(transformer_preds.predictions, axis=-1)

print("\n--- Upgraded DistilBERT Transformer Classification Report ---")
print(classification_report(true_labels, final_transformer_labels, target_names=dataset["train"].features["label"].names))

# 6. Save optimized artifacts for deployment
trainer.save_model("./production_transformer_model")
print("Optimization complete! Saved artifacts to directory: ./production_transformer_model")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Stage 3: Fine-Tuning DistilBERT Transformer ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.801875,0.243803,0.917500
2,0.203285,0.172734,0.934500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Upgraded DistilBERT Transformer Classification Report ---
              precision    recall  f1-score   support

     sadness       0.97      0.95      0.96       581
         joy       0.95      0.94      0.94       695
        love       0.80      0.87      0.83       159
       anger       0.91      0.93      0.92       275
        fear       0.89      0.87      0.88       224
    surprise       0.71      0.74      0.73        66

    accuracy                           0.92      2000
   macro avg       0.87      0.88      0.88      2000
weighted avg       0.92      0.92      0.92      2000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Optimization complete! Saved artifacts to directory: ./production_transformer_model
